In [1]:
import pandas as pd
from neo4j import GraphDatabase
from getpass import getpass
from pathlib import Path

In [2]:
# 1. Konfigurasi koneksi Neo4j
URI = "bolt://127.0.0.1:7687"
USERNAME = "neo4j"
PASSWORD = getpass("Masukkan password Neo4j: ")

In [3]:
# 2. Membaca dataset
BATCH_SIZE = 5000

FILES = [
    (Path("data/processed/events_P1_training.csv"), "P1"),
    (Path("data/processed/events_P2_testing.csv"), "P2"),
]

In [4]:
# 3. Query untuk menyimpan aktivitas ke graph
QUERY = """
UNWIND $batch AS row

MERGE (v:Visitor {visitorId: row.visitorid})
MERGE (i:Item {itemId: row.itemid})

FOREACH (_ IN CASE WHEN row.event = 'view'
    THEN [1] ELSE [] END |
    CREATE (v)-[:VIEWED {
        timestamp: row.timestamp,
        fold: row.fold
    }]->(i)
)

FOREACH (_ IN CASE WHEN row.event = 'addtocart'
    THEN [1] ELSE [] END |
    CREATE (v)-[:ADDED_TO_CART {
        timestamp: row.timestamp,
        fold: row.fold
    }]->(i)
)

FOREACH (_ IN CASE WHEN row.event = 'transaction'
    THEN [1] ELSE [] END |
    CREATE (v)-[:BOUGHT {
        timestamp: row.timestamp,
        transactionId: row.transactionid,
        fold: row.fold
    }]->(i)
)
"""

In [6]:
def main():
    for csv_path, _ in FILES:
        if not csv_path.exists():
            raise FileNotFoundError(
                f"File tidak ditemukan: {csv_path.resolve()}"
            )

    driver = GraphDatabase.driver(
        URI,
        auth=(USERNAME, PASSWORD)
    )

    try:
        with driver.session(database="neo4j") as session:
            result = session.run("RETURN 1 AS angka")
            print("Koneksi Neo4j berhasil:", result.single()["angka"])

            # Constraint untuk menjaga keunikan identitas node.
            session.run("""
                CREATE CONSTRAINT visitor_id_unique IF NOT EXISTS
                FOR (v:Visitor) REQUIRE v.visitorId IS UNIQUE
            """).consume()

            session.run("""
                CREATE CONSTRAINT item_id_unique IF NOT EXISTS
                FOR (i:Item) REQUIRE i.itemId IS UNIQUE
            """).consume()

        for csv_path, fold_name in FILES:
            print(f"\nMemulai impor {fold_name}: {csv_path}")

            total = 0

            for chunk in pd.read_csv(
                csv_path,
                chunksize=BATCH_SIZE
            ):
                records = []

                for row in chunk.itertuples(index=False):
                    transaction_id = (
                        None
                        if pd.isna(row.transactionid)
                        else int(row.transactionid)
                    )

                    records.append({
                        "timestamp": int(row.timestamp),
                        "visitorid": int(row.visitorid),
                        "event": row.event,
                        "itemid": int(row.itemid),
                        "transactionid": transaction_id,
                        "fold": fold_name
                    })

                with driver.session(database="neo4j") as session:
                    session.run(
                        QUERY,
                        batch=records
                    ).consume()

                total += len(records)
                print(
                    f"{fold_name}: {total:,} "
                    "aktivitas diproses"
                )

            print(f"Impor {fold_name} selesai: {total:,} aktivitas")

        print("\nSemua impor selesai.")

    finally:
        driver.close()

if __name__ == "__main__":
    main()

Koneksi Neo4j berhasil: 1

Memulai impor P1: data\processed\events_P1_training.csv
P1: 5,000 aktivitas diproses
P1: 10,000 aktivitas diproses
P1: 15,000 aktivitas diproses
P1: 20,000 aktivitas diproses
P1: 25,000 aktivitas diproses
P1: 30,000 aktivitas diproses
P1: 35,000 aktivitas diproses
P1: 40,000 aktivitas diproses
P1: 45,000 aktivitas diproses
P1: 50,000 aktivitas diproses
P1: 55,000 aktivitas diproses
P1: 60,000 aktivitas diproses
P1: 65,000 aktivitas diproses
P1: 70,000 aktivitas diproses
P1: 75,000 aktivitas diproses
P1: 80,000 aktivitas diproses
P1: 85,000 aktivitas diproses
P1: 90,000 aktivitas diproses
P1: 95,000 aktivitas diproses
P1: 100,000 aktivitas diproses
P1: 105,000 aktivitas diproses
P1: 110,000 aktivitas diproses
P1: 115,000 aktivitas diproses
P1: 120,000 aktivitas diproses
P1: 125,000 aktivitas diproses
P1: 130,000 aktivitas diproses
P1: 135,000 aktivitas diproses
P1: 140,000 aktivitas diproses
P1: 145,000 aktivitas diproses
P1: 150,000 aktivitas diproses
P1: 155